In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import(
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import transformers
from transformers import(
    BertTokenizer,
    BertModel,
    )

from tqdm.notebook import tqdm

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [4]:
df=pd.read_csv("/content/train.csv")
test=pd.read_csv("/content/test.csv")

In [5]:
df

,Id,Title,Abstract,Categories
0,9707,Axiomatic Aspects of Default Inference,This paper studies axioms for nonmonotonic con...,['cs.LO']
1,24198,On extensions of group with infinite conjugacy...,We characterize the group property of being wi...,['math.GR']
2,35766,An Analysis of Complex-Valued CNNs for RF Data...,Recent deep neural network-based device classi...,"['cs.LG', 'cs.IT', 'eess.SP', 'math.IT']"
3,14322,On the reconstruction of the drift of a diffus...,The problem of reconstructing the drift of a d...,"['math.PR', 'math.ST', 'stat.TH']"
4,709,Three classes of propagation rules for GRS and...,"In this paper, we study the Hermitian hulls of...","['cs.IT', 'math.IT']"
...,...,...,...,...
51205,46266,Generalized Fourier Integral Operators on spac...,Generalized Fourier integral operators (FIOs) ...,['math.AP']
51206,55910,Weakly-Supervised 3D Visual Grounding based on...,Learning to ground natural language queries to...,"['cs.CV', 'cs.CL']"
51207,10971,Strongly pseudoconvex handlebodies,We give an explicit construction of special st...,['math.CV']
51208,30982,Improving End-to-End Speech Processing by Effi...,Training a high performance end-to-end speech ...,"['cs.CL', 'cs.LG', 'cs.SD', 'eess.AS']"


In [6]:
train_data, val_data= train_test_split(df,test_size=0.1,random_state=42)

In [7]:
datatype=print(f"{df['Id'].map(type)[0]}','{df['Title'].map(type)[0]}','{df['Abstract'].map(type)[0]}','{df['Categories'].map(type)[0]}")

<class 'int'>','<class 'str'>','<class 'str'>','<class 'str'>


In [8]:
def remove_extra_spaces(text):
  string=text.split()
  return " ".join(string)

def convert_to_list(text):
  text=text[1:-1]
  text=text.replace(" ","")
  text=text.replace("'","")
  text=text.split(",")
  return text


In [9]:
train_category=train_data['Categories'].apply(convert_to_list).tolist() #pandas Series to python list
val_category=val_data['Categories'].apply(convert_to_list).tolist()


In [10]:
labels=['math.AT', 'stat.AP', 'cs.AR', 'math.QA', 'q-bio.MN', 'eess.AS','eess.IV', 'stat.ME', 'econ.GN',
            'eess.SP', 'q-fin.RM', 'cs.LG', 'cs.CR', 'q-bio.BM', 'q-fin.GN', 'q-fin.MF', 'q-fin.PR', 'math.CV',
            'cs.LO', 'econ.TH', 'math.CO', 'cs.AI', 'math.AC', 'q-bio.CB','q-fin.CP', 'cs.CL', 'cs.DC', 'math.LO',
            'math.NT', 'cs.SD', 'q-fin.TR','cs.CV', 'stat.ML', 'q-fin.EC', 'econ.EM', 'cs.CE', 'stat.CO','math.PR',
            'q-bio.NC', 'math.AP', 'cs.OS', 'cs.NI', 'cs.IT', 'cs.PL', 'cs.GT', 'cs.DM', 'math.IT', 'cs.SE', 'cs.RO',
            'stat.TH', 'cs.DB','math.ST', 'q-bio.GN', 'q-fin.PM', 'q-bio.TO', 'math.GR', 'cs.IR']


In [11]:
mlb=MultiLabelBinarizer(classes=labels)
train_label=mlb.fit_transform(train_category)
val_label=mlb.transform(val_category)

In [12]:
!pip install pylatexenc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 13.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=209036db6189badddd325d63eed28ae7126fc867972679b0ebdcb41b2a2f23e6
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [13]:
from pylatexenc.latex2text import LatexNodes2Text

In [14]:
import re

def remove_links(input_string):

    pattern1 = r'\\href\{.*?\}\{.*?\}'
    pattern2 = r'\\href\{.*?\}'
    pattern3 = r'\\url\{.*?\}'

    cleaned_string = re.sub(pattern1, '', input_string)
    cleaned_string = re.sub(pattern2, '', cleaned_string)
    cleaned_string = re.sub(pattern3, '', cleaned_string)

    return cleaned_string

def clean_text(text):
    text = remove_links(text)
    text = LatexNodes2Text().latex_to_text(text)
    text = remove_extra_spaces(text)
    text = re.sub(r'[^a-zA-Z0-9\s.,;:!?(){}\[\]<>+\-/*=%$&@#~≥\\_~`]', '', text)
    return text

In [15]:
Threshold=0.3
epochs=3

In [16]:
Bert_Tokenizer=BertTokenizer.from_pretrained("allenai/scibert_scivocab_cased")




vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

In [29]:
class MLCD(Dataset):
  def __init__(self,data,tokenizer,test,label=None):
    self.title=data['Title'].to_numpy()
    self.abstract=data['Abstract'].to_numpy()
    self.ids=data['Id'].to_numpy()
    self.label=label
    self.tokenizer=tokenizer
    self.test=test

  def __len__(self):
    return len(self.ids)

  def __getitem__(self,idx):
    encoding=self.tokenizer(
        self.title[idx],
        self.abstract[idx],
        max_length=512,
        add_special_tokens=True,
        padding='max_length',
        truncation=True,
        return_token_type_ids=True,#for2ndsentence
        return_attention_mask=True
    )
    '''token_type_ids = encoding.get(
    'token_type_ids',
    [0] * len(encoding['input_ids'])
)'''

    if not self.test:
      return {
          'textID' : self.ids[idx],
                'ids': torch.tensor(encoding['input_ids'], dtype=torch.long),
                'mask': torch.tensor(encoding['attention_mask'], dtype=torch.long),
                'token_type_ids': torch.tensor(encoding['token_type_ids'], dtype=torch.long),
                'targets': torch.tensor(self.label[idx],dtype=torch.float32)
      }
    else :
      return {
          'textID' : self.ids[idx],
                'ids': torch.tensor(encoding['input_ids'], dtype=torch.long),
                'mask': torch.tensor(encoding['attention_mask'], dtype=torch.long),
                'token_type_ids': torch.tensor(encoding['token_type_ids'], dtype=torch.long),
      }

In [18]:
train_dataset1=MLCD(train_data,train_label,Bert_Tokenizer,False)

val_dataset1=MLCD(val_data,val_label,Bert_Tokenizer,False)


In [19]:
train_dataloader1=DataLoader(train_dataset1,batch_size=32,shuffle=True)
val_dataloader1=DataLoader(val_dataset1,batch_size=32,shuffle=False)


In [20]:


class BertNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.BertModel=BertModel.from_pretrained('allenai/scibert_scivocab_cased')
    self.l1=nn.Linear(768,1024)
    self.activ=nn.GELU()
    self.dropout=nn.Dropout(0.1)
    self.l2=nn.Linear(1024,57)

  def forward(self,inputIds,attention_mask,token_type_ids):
      out=self.BertModel(input_ids=inputIds,attention_mask=attention_mask,token_type_ids=token_type_ids)
      out=out.last_hidden_state[:,0,] #bertoutput=(batch_Size,seqlen,768)
      out=self.l1(out)
      out=self.activ(out)
      out=self.dropout(out)
      out=self.l2(out)

      return out

In [21]:

model2=BertNN().to(device)


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allenai/scibert_scivocab_cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
criterion=nn.BCEWithLogitsLoss()
def loss_fn(output,target):
  return criterion(output,target)

optimizer2=optim.AdamW(params=model2.parameters(),lr=2e-5,weight_decay=0.01)

In [23]:


def Bert_train(train_loader,model,loss_fn,optimizer,device):
  losses=[]
  model.train()
  progress=tqdm(train_loader,total=len(train_loader))
  for _,data in enumerate(progress):
    ids=data['ids'].to(device)
    mask=data['mask'].to(device)
    token_type_ids=data['token_type_ids'].to(device)
    targets=data['targets'].to(device)
    outputs=model(ids,mask,token_type_ids)
    loss=loss_fn(outputs,targets)
    losses.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  return np.mean(losses)

In [24]:

train_losses2 = []
best_dict2 = None
best_loss2 = np.inf
for ep in range(epochs):
   print(f" Epoch {ep+1}")
   bert_loss=Bert_train(train_dataloader1, model2, loss_fn, optimizer2,device)
   if bert_loss < best_loss2:
        best_loss2 = bert_loss
        best_dict2 = model2.state_dict()
        checkpoint = {'model': model2, 'state_dict': model2.state_dict()}
        torch.save(checkpoint, 'checkpoint.pth')
   train_losses2.append(bert_loss)
   print(f"Epoch {ep + 1} - Train Loss {bert_loss:.4f}\n")


 Epoch 1


  0%|          | 0/1441 [00:00<?, ?it/s]

Epoch 1 - Train Loss 0.0896

 Epoch 2


  0%|          | 0/1441 [00:00<?, ?it/s]

Epoch 2 - Train Loss 0.0459

 Epoch 3


  0%|          | 0/1441 [00:00<?, ?it/s]

Epoch 3 - Train Loss 0.0385



In [25]:


def Bert_val(val_loader,model,device):
  finalbert=[]
  finalberttarget=[]
  model.eval()
  progress=tqdm(val_loader,total=len(val_loader))
  with torch.no_grad():
    for _,data in enumerate(progress):
      ids=data['ids'].to(device)
      mask=data['mask'].to(device)
      token_type_ids=data['token_type_ids'].to(device)
      target=data['targets'].to(device)
      output=model(ids,mask,token_type_ids)
      output=torch.sigmoid(output).cpu().detach().tolist()
      finalberttarget.extend(target.cpu().detach().tolist())
      outlist=[]
      for out in output:
        yo=[]
        for term in out:
          if term>=Threshold:
            yo.append(1)
          else:
            yo.append(0)
        outlist.append(yo)
      finalbert.extend(outlist)
  return finalbert, finalberttarget


In [26]:
Bertoutput,Berttarget=Bert_val(val_dataloader1,model2,device)
f1_score2=f1_score(Berttarget,Bertoutput,average='macro')
print(f"Macro_f1_score_is={f1_score2}")

  0%|          | 0/161 [00:00<?, ?it/s]

Macro_f1_score_is=0.6787051145669526


In [37]:
def Bert_test(test_loader, model):
    model.eval()
    progress = tqdm(test_loader, total=len(test_loader))
    with torch.no_grad():
        pred_dict = {}
        for _, data in enumerate(progress):
            textid = data['textID']
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            outputs = model(ids, mask, token_type_ids)
            outlist = torch.sigmoid(outputs).cpu().detach().numpy().tolist()
            one_hot_out = []
            for outl in outlist:
                yo = []
                for term in outl:
                    if(term >= Threshold):
                        yo.append(1)
                    else:
                        yo.append(0)
                one_hot_out.append(yo)
            pred_dict[textid] = one_hot_out


    return pred_dict

In [32]:
test_dataset = MLCD(test,Bert_Tokenizer,test = True)
test_loader  = DataLoader(test_dataset, batch_size = 32, num_workers = 4,shuffle=False)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [38]:
pred_dict = Bert_test(test_loader, model2)

  0%|          | 0/343 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
